In [15]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.rand(size=(2, 4),requires_grad=True)
net(X)

tensor([[-0.0295],
        [ 0.0215]], grad_fn=<AddmmBackward0>)

In [30]:
print(net[2].state_dict())

OrderedDict({'weight': tensor([[-0.2060, -0.1420,  0.0558, -0.2709,  0.3077, -0.2018,  0.2673,  0.2134]]), 'bias': tensor([0.0598])})


In [31]:
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([0.0598], requires_grad=True)
tensor([0.0598])


In [32]:
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


In [33]:
net.state_dict()['2.bias'].data

tensor([0.0598])

In [34]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        # 在这里嵌套
        net.add_module(f'block {i}', block1())
    return net

rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
rgnet(X)

tensor([[0.4154],
        [0.4153]], grad_fn=<AddmmBackward0>)

In [35]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


In [37]:
rgnet[0][1][0].bias.data

tensor([-0.2447,  0.3064, -0.0698, -0.0703,  0.4231, -0.1845,  0.1684, -0.0531])

In [39]:
rgnet[0][1][2].bias.data

tensor([ 0.0034, -0.2031,  0.2306, -0.0916])

In [49]:
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)
net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([ 0.0122, -0.0150,  0.0067, -0.0027]), tensor(0.))

In [50]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)
net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

In [51]:
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)

net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)

tensor([ 0.6929, -0.1675, -0.1176, -0.0777])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


In [62]:
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape)
                        for name, param in m.named_parameters()][0])
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(my_init)
net[0].weight[:]

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[ 8.4942,  8.2339, -7.5059,  0.0000],
        [ 5.7139, -0.0000, -9.8115,  0.0000],
        [-8.8827, -0.0000,  5.6889, -0.0000],
        [ 7.7047,  7.1525, -5.1535, -6.0707],
        [ 0.0000,  0.0000, -0.0000,  0.0000],
        [-0.0000,  5.8145, -6.3444,  7.1482],
        [ 8.1528,  0.0000,  7.1172, -7.6501],
        [ 6.2445,  0.0000,  0.0000, -5.1802]], grad_fn=<SliceBackward0>)

In [63]:
net[0].weight.data[:] += 1
net[0].weight.data[0, 0] = 42
net[0].weight.data[0]

tensor([42.0000,  9.2339, -6.5059,  1.0000])

In [64]:
# 我们需要给共享层一个名称，以便可以引用它的参数
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.Linear(8, 1))
net(X)
# 检查参数是否相同
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
# 确保它们实际上是同一个对象，而不只是有相同的值
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])
